## Dependencies

In [1]:
## libraries
import sys
import logging
import pandas as pd
from pathlib import Path

## path
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

## modules
from src.estimators.factories import load_estimators
from src.data.builders import (
    load_processed_data, 
    load_falsified_data
)
from src.evaluators.falsifying import (
    eval_falsified_frontier,
    eval_falsified_alignment,
    eval_falsified_consensus,
    stat_falsified_test,
    stat_falsified_summary,
)
from src.evaluators.metrics import (
    FRONTIER_METRICS,
    CONSENSUS_METRICS
)
from src.evaluators.config import (
    FEAT_X, 
    FEAT_Z, 
    TARGET
)

## Initialization

In [2]:
## load data and models
_disable = logging.root.manager.disable
logging.disable(logging.INFO)
try:
    data_proc = load_processed_data()
    data_fals = load_falsified_data()
    models = load_estimators()
finally:
    logging.disable(_disable)

## view data shape
n_obs_proc, n_feat_proc = data_proc.shape
print("Original Data")
print(f"    Features: {n_feat_proc}")
print(f"    Observations: {n_obs_proc}")

print("\nFalsified Data")
for method, df in data_fals.items():
    n_obs_fals, n_feat_fals = df.shape
    print(f"  {method}:")
    print(f"    Features: {n_feat_fals}")
    print(f"    Observations: {n_obs_fals}")

Original Data
    Features: 32
    Observations: 25

Falsified Data
  random_generate:
    Features: 32
    Observations: 25
  target_remap:
    Features: 32
    Observations: 25
  vector_generate:
    Features: 32
    Observations: 25


## Frontier Falsifiability Test
This section evaluates whether models produce better frontier envelope metrics on original data than on falsified data. Under the frozen protocol, models trained on original data are evaluated on falsified inputs. Under the retrain protocol models are retrained on falsified data and compared to the original-data baseline.

In [12]:
## test falsified frontier
if "results_falsified_frontier" not in globals():
    results_falsified_frontier = eval_falsified_frontier(
        data_proc = data_proc,
        data_fals = data_fals,
        models = models,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
        group = 'domain',
        n_repeat = 30,
        random_state = 42
    )

In [13]:
## wilcoxon signed-rank test for falsified frontier
results_table_frontier = stat_falsified_test(
    results = results_falsified_frontier,
    feat_value = ["ei"],
    feat_pairs = ["model", "group"]
 )

display(results_table_frontier)

=== Falsifiability: Original vs Falsified Median ei (n = 45) ===
H₁: Original data produces higher median ei than falsified data
*** p < 0.001, ** p < 0.01, * p < 0.05


Median ei (Original) Median ei (Falsified)  \
Track   Method                                                       
frozen  target_remap                  0.7692                0.6862   
        random_generate               0.7692                0.7283   
        vector_generate               0.7692                0.7161   
retrain target_remap                  0.7692                0.7291   
        random_generate               0.7692                0.7267   
        vector_generate               0.7692                0.7245   

                        Median Δ ei Positive Δ Wilcoxon W+ Rank-biserial r  \
Track   Method                                                               
frozen  target_remap         0.0287     0.6444         744          0.4377   
        random_generate      0.0532     0.7778         877          0.6947   
        vector_generate      0.0376     0.7333         870          0.6812   
retrain target_remap         0.0385        0.6         763          0.4744   
        random_generate      0.0483     0.7556         787          0.5208   
        vector_generate       0.046        0.8         799           0.544   

                        One-sided p Holm-adjusted p Sig.  
Track   Method                                            
frozen  target_remap         0.0049          0.0049   **  
        random_generate         0.0          0.0001  ***  
        vector_generate         0.0          0.0001  ***  
retrain target_remap         0.0025          0.0049   **  
        random_generate      0.0009          0.0028   **  
        vector_generate      0.0006          0.0023   **

In [14]:
## median falsified frontier metrics
frontier_summary = stat_falsified_summary(
    results = results_falsified_frontier,
    metrics = FRONTIER_METRICS
)

display(frontier_summary)

vr      mv      ms      ea      ei
track    method                                              
original original         0.0  0.0000  4.5428  0.7634  0.7692
frozen   target_remap     0.2  0.4433  5.0140  0.8050  0.6862
         random_generate  0.0  0.0000  7.8469  1.1106  0.7283
         vector_generate  0.0  0.0000  6.8283  1.1265  0.7161
retrain  target_remap     0.0  0.0000  5.6599  0.8360  0.7291
         random_generate  0.0  0.0000  5.8010  0.8752  0.7267
         vector_generate  0.0  0.0000  5.2695  0.8656  0.7245

## Target-Alignment Tests
This section tests whether original data produces stronger alignment between model predictions and the observed target than falsified data. 
It measures global rank and correlation agreement using cross-validated predictions under both frozen and retrained protocols. It treats each falsification method as an independent perturbation of the input signal.

In [6]:
## test falsified target alignment
if "results_falsified_alignment" not in globals():
    results_falsified_alignment = eval_falsified_alignment(
        data_proc = data_proc,
        data_fals = data_fals,
        models = models,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
        group = "domain",
        n_repeat = 30,
        random_state = 42
    )

In [7]:
## wilcoxon signed-rank test for falsified target alignment
results_table_alignment = stat_falsified_test(
    results = results_falsified_alignment,
    feat_value = ["rho"],
    feat_pairs = ["model", "group"]
)

display(results_table_alignment)

=== Falsifiability: Original vs Falsified Median rho (n = 9) ===
H₁: Original data produces higher median rho than falsified data
*** p < 0.001, ** p < 0.01, * p < 0.05


Median rho (Original) Median rho (Falsified)  \
Track   Method                                                         
frozen  target_remap                   0.6638                -0.2094   
        random_generate                0.6638                -0.1638   
        vector_generate                0.6638                 0.0423   
retrain target_remap                   0.6638                -0.0346   
        random_generate                0.6638                -0.1315   
        vector_generate                0.6638                -0.2715   

                        Median Δ rho Positive Δ Wilcoxon W+ Rank-biserial r  \
Track   Method                                                                
frozen  target_remap            0.84        1.0          45             1.0   
        random_generate       0.9401        1.0          45             1.0   
        vector_generate       0.5384     0.8889          43          0.9111   
retrain target_remap          0.6585        1.0          45             1.0   
        random_generate       0.7554        1.0          45             1.0   
        vector_generate       0.8862        1.0          45             1.0   

                        One-sided p Holm-adjusted p Sig.  
Track   Method                                            
frozen  target_remap          0.002          0.0117    *  
        random_generate       0.002          0.0117    *  
        vector_generate      0.0059          0.0117    *  
retrain target_remap          0.002          0.0117    *  
        random_generate       0.002          0.0117    *  
        vector_generate       0.002          0.0117    *

In [8]:
## median falsified target-alignment metrics
alignment_summary = stat_falsified_summary(
    results = results_falsified_alignment,
    metrics = CONSENSUS_METRICS
)

display(alignment_summary)

r     rho     tau     rbo     dcr
track    method                                                 
original original         0.6523  0.6638  0.4467  0.5792  0.6464
frozen   target_remap    -0.1334 -0.2094 -0.1400  0.2908  0.3393
         random_generate -0.2021 -0.1638 -0.1267  0.3090  0.3973
         vector_generate  0.0395  0.0423  0.0333  0.3566  0.3193
retrain  target_remap     0.0173 -0.0346 -0.0200  0.2986  0.3007
         random_generate -0.1958 -0.1315 -0.0800  0.3438  0.3195
         vector_generate -0.3435 -0.2715 -0.2067  0.3224  0.4355

## Pairwise Consensus Test
This section compares fitted model frontiers to each other rather than comparing each model to the target. It evaluates whether inter-model frontier agreement declines under falsification for both the frozen and retrained protocols. It treats pairwise consensus as a fitted-frontier agreement analysis rather than a predictive resampling test.

In [9]:
## test falsified pairwise consensus
if "results_falsified_consensus" not in globals():
    results_falsified_consensus = eval_falsified_consensus(
        data_proc = data_proc,
        data_fals = data_fals,
        models = models,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
        n_repeat = 30,
        random_state = 42
    )

In [10]:
## wilcoxon signed-rank test for falsified pairwise consensus
results_table_pairwise = stat_falsified_test(
    results = results_falsified_consensus,
    feat_value = ["rho"],
    feat_pairs = ["model_i", "model_j", "group"]
)

display(results_table_pairwise)

=== Falsifiability: Original vs Falsified Median rho (n = 36) ===
H₁: Original data produces higher median rho than falsified data
*** p < 0.001, ** p < 0.01, * p < 0.05


Median rho (Original) Median rho (Falsified)  \
Track   Method                                                         
frozen  target_remap                   0.9615                 0.9615   
        random_generate                0.9615                 0.1227   
        vector_generate                0.9615                 0.4617   
retrain target_remap                   0.9615                 0.3546   
        random_generate                0.9615                 0.5331   
        vector_generate                0.9615                 0.7454   

                        Median Δ rho Positive Δ Wilcoxon W+ Rank-biserial r  \
Track   Method                                                                
frozen  target_remap             0.0        0.0           -               -   
        random_generate       0.8669     0.9722         630             1.0   
        vector_generate       0.4965        1.0         666             1.0   
retrain target_remap          0.6292        1.0         666             1.0   
        random_generate         0.41     0.8611         642          0.9279   
        vector_generate       0.1854     0.9167         591          0.9866   

                        One-sided p Holm-adjusted p Sig.  
Track   Method                                            
frozen  target_remap              -               -    -  
        random_generate         0.0             0.0  ***  
        vector_generate         0.0             0.0  ***  
retrain target_remap            0.0             0.0  ***  
        random_generate         0.0             0.0  ***  
        vector_generate         0.0             0.0  ***

In [11]:
## median pairwise consensus metrics
pairwise_summary = stat_falsified_summary(
    results = results_falsified_consensus,
    metrics = CONSENSUS_METRICS
)

display(pairwise_summary)

r     rho     tau     rbo     dcr
track    method                                                 
original original         0.9695  0.9615  0.8467  0.8987  0.9617
frozen   target_remap     0.9695  0.9615  0.8467  0.8987  0.9617
         random_generate  0.2926  0.1227  0.0733  0.4410  0.3515
         vector_generate  0.4663  0.4617  0.3267  0.5493  0.5065
retrain  target_remap     0.3867  0.3546  0.2633  0.5325  0.4752
         random_generate  0.5225  0.5331  0.4033  0.5318  0.5978
         vector_generate  0.7435  0.7454  0.5767  0.7752  0.7396